# Exploitation notebook

In [13]:
import gym
import highway_env
from stable_baselines3 import PPO

# Evaluation
from highway_env.envs.common.evaluate import PrintMetrics
import random

# Visualisation   
from ipywidgets import interact, widgets
import glob
import numpy as np

In [14]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO','MACOS', 'WINDOWS', 'UBUNTU_MICRO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'MACOS', 'WINDOWS', 'UBUNTU_M…

<function __main__.f(desired_os)>

In [15]:
ABS_PATH = '/Users/fornerispighetti/HighwayDRL' if os_id.value == "MACOS" else 'C:/Users/pigo/Desktop/HighwayDRL'
if os_id.value == 'UBUNTU_MICRO':
    ABS_PATH = '/home/utente1/Desktop/PighettiForneris/HighwayDRL'
elif os_id.value == 'UBUNTU_DELL':
    ABS_PATH = '/home/elios/pighetti/HighwayDRL'
elif os_id.value == 'UBUNTU_PIGO':
    ABS_PATH = '/home/pigo/HighwayDRL'

### Choose the environment to exploit the model in

In [16]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0', 'highway-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

### Choose the model

In [17]:
model_id = widgets.Text()
def m(input_model):
    model_id.value = str(input_model)
interact(m, input_model=glob.glob(f"{ABS_PATH}/training_output/models/*.zip"))

interactive(children=(Dropdown(description='input_model', options=('/home/pigo/HighwayDRL/training_output/mode…

<function __main__.m(input_model)>

In [18]:
explicit_env = widgets.Checkbox(
    value=False,
    description='explicit env?',
    disabled=False,
    indent=False
)
display(explicit_env)

Checkbox(value=False, description='explicit env?', indent=False)

In [20]:
random_agent = widgets.Checkbox(
    value=False,
    description='random behaviour?',
    disabled=False,
    indent=False
)
display(random_agent)

Checkbox(value=False, description='random behaviour?', indent=False)

In [22]:
render = widgets.Checkbox(
    value=False,
    description='render scene?',
    disabled=False,
    indent=False
)
display(render)

Checkbox(value=False, description='render scene?', indent=False)

In [23]:
# Number of exploitation episodes
episode_num = 10
metricObj = PrintMetrics()
# metric counters

expl_envs = ['dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','multipleO-dm-env-v0']

if(explicit_env.value):
    env = gym.make(env_id.value)
    random_env = env_id.value
    env.configure({
        "screen_width":1500
    })
    
if not random_agent.value:
        model = PPO.load(model_id.value)
        print('using PPO')


for episode in range(episode_num):
    if(not explicit_env.value):
        random_env = random.choice(expl_envs)
        env = gym.make(random_env)
        env.configure({
            "screen_width":1500,
            "screen_height":250
        })
        
    obs, done = env.reset(), False

    # Reset metric counters at the beginning of each episode
    decision_change_num, left_lane_change_num, right_lane_change_num, episode_reward = 0, 0, 0, 0
    accelerations, decelerations, speeds = [], [], []
    last_lane_idx = 1000
    last_action = ""
    
    while not done:
        if(random_agent.value):
            action = env.random_action()
            obs, reward, done, info = env.step(action)
        else:
            action, _states = model.predict(obs)
            obs, reward, done, info = env.step(int(action))
        
        episode_reward += reward
            
        if(render.value):
            env.render()

        # Update metric counters        
        if last_action != env.vehicle.current_action and last_action != "":
            decision_change_num += 1
        last_action = env.vehicle.current_action    
        
        speeds.append(env.vehicle.speed)
        
        if env.vehicle.throttle > 0:
            accelerations.append(env.vehicle.throttle)
        else:
            decelerations.append(env.vehicle.throttle)

        if last_lane_idx != env.vehicle.lane_index[2]:
            if last_lane_idx > env.vehicle.lane_index[2] and last_lane_idx != 1000:
                left_lane_change_num += 1
            else:
                right_lane_change_num += 1
            last_lane_idx = env.vehicle.lane_index[2]

    episode_duration = (env.steps/env.config["policy_frequency"]*env.config["simulation_frequency"])/30
    decision_change_rate = decision_change_num / episode_duration
    left_lane_change_rate = left_lane_change_num / episode_duration
    right_lane_change_rate = right_lane_change_num / episode_duration
    mean_speed = np.mean(speeds)
    km_travelled = (mean_speed * episode_duration)/1000
    mean_acceleration = np.mean(accelerations)
    mean_deceleration = np.mean(decelerations)
    metricObj.printEpisode(random_env, km_travelled, decision_change_num, decision_change_rate, left_lane_change_num, left_lane_change_rate,\
        right_lane_change_num, right_lane_change_rate, mean_speed*3.6, mean_acceleration, mean_deceleration, env.vehicle.crashed, episode_duration, episode+1)

env.close()
if random_agent.value:
    csv_id = 'random_agent'
else:
    csv_id = model_id.value[model_id.value.find('ppo'):model_id.value.find('.zip')]
    
metricObj.printRecap(f"{ABS_PATH}/exploitation/eval_metrics/", csv_id)

/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(


using standard PPO


/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highwa


episode 1: dm-env-v0 ended, metrics:             
	episode duration: 60 seconds,
	collision? NO 
	km travelled: 2.03,             
	decision changes: 53, 
	mean speed: 121.65 km/h, 
	mean acceleration: nan m/s, 
	mean deceleration: -4.524 m/s,             
	decision change rate: 0.88, 
	left lane changes: 2, 
	left lane change rate: 0.03             
	right lane changes: 3, 
	right lane change rate: 0.05

TEST - Reward: 69.32296922273848


/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highwa

...

/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highwa


episode 9: dm-env-v0 ended, metrics:             
	episode duration: 60 seconds,
	collision? NO 
	km travelled: 1.28,             
	decision changes: 20, 
	mean speed: 76.74 km/h, 
	mean acceleration: 2.85 m/s, 
	mean deceleration: -3.992 m/s,             
	decision change rate: 0.33, 
	left lane changes: 1, 
	left lane change rate: 0.02             
	right lane changes: 0, 
	right lane change rate: 0.0

TEST - Reward: 18.35521967737338


/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highwa


episode 10: dm-env-v0 ended, metrics:             
	episode duration: 7 seconds,
	collision? YES 
	km travelled: 0.22,             
	decision changes: 7, 
	mean speed: 113.53 km/h, 
	mean acceleration: 4.434 m/s, 
	mean deceleration: -4.036 m/s,             
	decision change rate: 1.0, 
	left lane changes: 2, 
	left lane change rate: 0.29             
	right lane changes: 2, 
	right lane change rate: 0.29

TEST - Reward: 3.8464552939946004


/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/observation.py:215: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(pd.DataFrame.from_records(
/home/pigo/HighwayDRL/highway_env/envs/common/evaluate.py:26: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  self.episode_df = self.episode_df.append(series)
